# Faithfulness summary

Set `DEBUG = True` in the setup cell to keep any new outputs under `Temp`; the source faithfulness CSV is read from the cloned repo results.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def run_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        message = [
            f"Command failed: {' '.join(map(str, command))}",
            f"Return code: {result.returncode}",
        ]
        if result.stdout:
            message.append(f"STDOUT:\n{result.stdout}")
        if result.stderr:
            message.append(f"STDERR:\n{result.stderr}")
        raise RuntimeError("\n".join(message))
    return result


def resolve_working_root():
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    if Path("/content").exists():
        return Path("/content")
    return Path.cwd().resolve()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = resolve_working_root()
    clone_dir = working_root / REPO_NAME

    if (clone_dir / PACKAGE_PATH).exists():
        return clone_dir.resolve()

    if clone_dir.exists() and (clone_dir / ".git").exists():
        run_command(["git", "-C", str(clone_dir), "pull", "--ff-only"])
        if (clone_dir / PACKAGE_PATH).exists():
            return clone_dir.resolve()

    if clone_dir.exists() and not (clone_dir / PACKAGE_PATH).exists():
        clone_dir = working_root / f"{REPO_NAME}_repo"

    if not clone_dir.exists():
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])

    if not (clone_dir / PACKAGE_PATH).exists():
        raise FileNotFoundError(
            f"El repositorio clonado no contiene {PACKAGE_PATH}: {clone_dir}"
        )

    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    run_command([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


PROJECT_ROOT: /Users/diego/Documents/STF-KernelSHAP
INSTALL_REQUIREMENTS: False


In [2]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
REPO_MODELS_DIR = paths.repo_models_dir
REPO_RESULTS_DIR = paths.repo_results_dir
REPO_FIGURES_DIR = paths.repo_figures_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


Repository root: /Users/diego/Documents/STF-KernelSHAP
Dataset root: /Users/diego/Documents/STF-KernelSHAP/MI_TDAH_Dataset
Debug mode: True
Results dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Results
Figures dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Figures
Output models dir: /Users/diego/Documents/STF-KernelSHAP/Temp/Models


In [3]:
import pandas as pd

from stf_kernelshap.reporting.faithfulness_summary import (
    aggregate_auc_by_window_model_method,
    compute_average_rank_by_window_metric,
    compute_faithfulness_auc_summary,
    get_best_only_by_window,
    print_rankings_for_window,
)


In [4]:
plot_ready_csv = REPO_RESULTS_DIR / "faithfulness_selected_relevance" / "faithfulness_selected_relevance_score_plot_ready.csv"
df_plot_ready = pd.read_csv(plot_ready_csv)

df_auc = compute_faithfulness_auc_summary(df_plot_ready)
df_auc_summary = aggregate_auc_by_window_model_method(df_auc)
df_average_rank = compute_average_rank_by_window_metric(df_auc_summary)


In [5]:
rankings = {
    window_name: print_rankings_for_window(df_auc_summary, window_name)
    for window_name in ("2.5-5", "0-7", "TDAH")
}

best_by_window = {
    window_name: get_best_only_by_window(df_auc_summary, window_name)
    for window_name in ("2.5-5", "0-7", "TDAH")
}


WINDOW: 2.5-5

MoRF AUC ranking
Criterio: menor MoRF_accuracy_AUC_mean = mejor



,window,model,strategy,rank_MoRF,MoRF_accuracy_AUC_mean,MoRF_accuracy_AUC_std,n_cases
7,2.5-5,ShallowConvNet,IntegratedGradients,1,0.204587,0.046205,2
10,2.5-5,ShallowConvNet,Occlusion,2,0.306178,0.075306,2
8,2.5-5,ShallowConvNet,KernelSHAP,3,0.609201,0.149975,2
9,2.5-5,ShallowConvNet,LIME,4,0.616227,0.140038,2
11,2.5-5,ShallowConvNet,STF-KernelSHAP,5,0.633351,0.194292,2
6,2.5-5,ShallowConvNet,GradCAM++,6,0.790339,0.029259,2



ROAD AUC ranking
Criterio: mayor ROAD_accuracy_gap_AUC_mean = mejor



,window,model,strategy,rank_ROAD,ROAD_accuracy_gap_AUC_mean,ROAD_accuracy_gap_AUC_std,n_cases
10,2.5-5,ShallowConvNet,Occlusion,1,0.644913,0.118642,2
7,2.5-5,ShallowConvNet,IntegratedGradients,2,0.610073,0.286690,2
11,2.5-5,ShallowConvNet,STF-KernelSHAP,3,0.250026,0.255122,2
6,2.5-5,ShallowConvNet,GradCAM++,4,0.060617,0.041551,2
9,2.5-5,ShallowConvNet,LIME,5,0.007044,0.007776,2
8,2.5-5,ShallowConvNet,KernelSHAP,6,-0.000141,0.000200,2


WINDOW: 0-7

MoRF AUC ranking
Criterio: menor MoRF_accuracy_AUC_mean = mejor



...

,window,model,strategy,rank_ROAD,ROAD_accuracy_gap_AUC_mean,ROAD_accuracy_gap_AUC_std,n_cases
4,0-7,ShallowConvNet,Occlusion,1,0.476362,0.575805,2
1,0-7,ShallowConvNet,IntegratedGradients,2,0.469070,0.656137,2
5,0-7,ShallowConvNet,STF-KernelSHAP,3,0.454308,0.090600,2
0,0-7,ShallowConvNet,GradCAM++,4,0.085159,0.087677,2
3,0-7,ShallowConvNet,LIME,5,0.009112,0.002805,2
2,0-7,ShallowConvNet,KernelSHAP,6,0.000007,0.000010,2


WINDOW: TDAH

MoRF AUC ranking
Criterio: menor MoRF_accuracy_AUC_mean = mejor



,window,model,strategy,rank_MoRF,MoRF_accuracy_AUC_mean,MoRF_accuracy_AUC_std,n_cases
13,TDAH,T-GARNet,IntegratedGradients,1,0.107076,NaN,1
16,TDAH,T-GARNet,Occlusion,2,0.495808,NaN,1
17,TDAH,T-GARNet,STF-KernelSHAP,3,0.646051,NaN,1
14,TDAH,T-GARNet,KernelSHAP,4,0.689869,NaN,1
15,TDAH,T-GARNet,LIME,5,0.720626,NaN,1
12,TDAH,T-GARNet,GradCAM++,6,0.728681,NaN,1



ROAD AUC ranking
Criterio: mayor ROAD_accuracy_gap_AUC_mean = mejor



,window,model,strategy,rank_ROAD,ROAD_accuracy_gap_AUC_mean,ROAD_accuracy_gap_AUC_std,n_cases
13,TDAH,T-GARNet,IntegratedGradients,1,0.880763,NaN,1
17,TDAH,T-GARNet,STF-KernelSHAP,2,0.330597,NaN,1
16,TDAH,T-GARNet,Occlusion,3,0.326097,NaN,1
14,TDAH,T-GARNet,KernelSHAP,4,-0.025342,NaN,1
15,TDAH,T-GARNet,LIME,5,-0.038549,NaN,1
12,TDAH,T-GARNet,GradCAM++,6,-0.286078,NaN,1
